In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Shared telescope geometry, the danish-based donut simulator, and figure
# helpers live in the installed `shape_vs_intensity` package (pip install -e .).
import galsim.zernike as gzk
from shape_vs_intensity import config as C
from shape_vs_intensity import sim
from shape_vs_intensity.plotutils import use_style

In [ ]:
use_style()

# --- Shared figure geometry ----------------------------------------------
# The map_circles schematics use a frame half-width of DIAGRAM_L with an outer
# ring radius of 1, so the ring fills 1 / DIAGRAM_L of the panel.  We crop the
# simulated donuts to the same fraction (see show_donut) so the two rows match.
# Repo sign convention: extra-focal is the +defocalOffset / +Z4 side;
# intra-focal is the -Z4 side.  The simulated donuts below are intra-focal, and
# the schematics use cos(m*theta) modes with the intra-focal map dp = -grad(W)
# (see the paper's "Aberrations in Defocused Images" section).
DIAGRAM_L = 1.25  # map_circles frame half-width (outer ring radius = 1)
DONUT_OUTER_PX = 66.7  # measured outer radius of a full-defocus donut, in pixels
ARCSEC_PER_PX = C.PIXEL_SCALE / C.FOCAL_LENGTH * 206265  # plate scale


def show_donut(ax, img):
    """Show a donut on ``ax``, cropped to match the schematic framing.

    Crops so a full-defocus ring fills the same fraction of the panel as the
    map_circles schematic ring (outer radius / half-width = 1 / DIAGRAM_L), then
    hides the axis ticks.
    """
    crop = C.NPIX // 2 - int(round(DONUT_OUTER_PX * DIAGRAM_L))
    ax.imshow(img[crop:-crop, crop:-crop], origin="lower")
    ax.set(xticks=[], yticks=[])

In [ ]:
def map_circles(A, displacement, N=5, ax=None, center=True, **kwargs):
    """Map concentric pupil circles through a displacement field.

    Parameters
    ----------
    A : float
        Scale factor applied to the displacement.
    displacement : callable
        Function ``displacement(rho, theta) -> (dx, dy)`` giving the Cartesian
        ring displacement in outer-pupil-radius units.
    N : int, optional
        Number of pupil circles to draw.
    ax : matplotlib.axes.Axes, optional
        Axes on which to draw the circles.
    center : bool, optional
        If True, center the view on the mapped circles.
    **kwargs
        Additional keyword arguments passed to ``Axes.plot``.
    """
    kwargs = {"c": "C1", "lw": 0.5, "zorder": 10} | kwargs
    if ax is None:
        fig, ax = plt.subplots()

    theta = np.linspace(0, 2 * np.pi, 10_000)
    for r in np.linspace(C.EPS_RUBIN, 1, N):
        rho = r * np.ones_like(theta)
        x0 = rho * np.cos(theta)
        y0 = rho * np.sin(theta)
        dx, dy = displacement(rho, theta)
        ax.plot(x0 + A * dx, y0 + A * dy, **kwargs)

    if center:
        x0 = np.mean(ax.get_xlim())
        y0 = np.mean(ax.get_ylim())
    else:
        x0, y0 = 0, 0
    ax.set(
        xlim=(x0 - DIAGRAM_L, x0 + DIAGRAM_L),
        ylim=(y0 - DIAGRAM_L, y0 + DIAGRAM_L),
        aspect="equal",
        xticks=[],
        yticks=[],
    )


def zernike_displacement(j, eps=C.EPS_RUBIN):
    """Return the intra-focal ring displacement for annular Noll mode ``j``.

    A defocused image maps pupil rays by ``dp = -grad(W)`` (see the paper's
    "Aberrations in Defocused Images" section, intra-focal side).  Here ``W`` is
    the annular Zernike ``Z_j`` on the unit pupil with central obscuration
    ``eps`` -- the *same* wavefront family danish uses to render the donut -- so
    the schematic ring diagram and the simulated donut are driven by one
    identical wavefront instead of a separately hand-derived gradient.

    Parameters
    ----------
    j : int
        Noll index of the annular Zernike mode.
    eps : float, optional
        Fractional central obscuration (inner / outer pupil radius).

    Returns
    -------
    callable
        ``displacement(rho, theta) -> (dx, dy)`` in outer-pupil-radius units.
    """
    coef = np.zeros(j + 1)
    coef[j] = 1.0
    zern = gzk.Zernike(coef, R_outer=1.0, R_inner=eps)
    grad_x, grad_y = zern.gradX, zern.gradY

    def displacement(rho, theta):
        x = rho * np.cos(theta)
        y = rho * np.sin(theta)
        return -grad_x(x, y), -grad_y(x, y)

    return displacement

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Defocus\n$\\nu = 0,\\, m=0$")
axes[0, 1].set_title("Tilt\n$\\nu = 0,\\, m=1$")

# --- Top row: wavefront-circle schematics --------------------------------
# Every schematic maps the pupil circles by dp = -grad(Z_j), the intra-focal
# displacement for annular Noll mode j (paper Sec. 3).  Defocus (Z4): a positive
# Z4 aberration on the intra-focal (-Z4) side reduces |defocus|, so the rings
# shrink.  Tilt (Z2): -grad(Z2) is a uniform leftward (-x) shift.  The per-panel
# amplitudes are display scales chosen for visual clarity.
map_circles(0.018, zernike_displacement(4), ax=axes[0, 0])
map_circles(0.105, zernike_displacement(2), ax=axes[0, 1], center=False)

# --- Bottom row: simulated intra-focal donuts ----------------------------
# Defocus sets the ring *size*: the positive-defocus panel uses smaller
# intra-focal defocus so its ring is smaller, shown centred.  Tilt leaves the
# size unchanged but *shifts* the ring straight left, matching the Z2 schematic.
# The ``dx`` parameter is an image-plane shift, negative to mimic positive
# wavefront tilt under the -grad(W) map.
DEFOCUS_SHRINK = 0.20  # fractional defocus reduction on the intra-focal side
TILT_SHIFT = 0.18  # leftward (-x) image shift, in outer-radius units

donut_defocus = sim.simulate_donut(
    defocus=(1 - DEFOCUS_SHRINK) * C.DEFOCUS_Z4, defocal_type="intra"
)
donut_tilt = sim.simulate_donut(
    dx=-TILT_SHIFT * DONUT_OUTER_PX * ARCSEC_PER_PX, defocal_type="intra"
)

show_donut(axes[1, 0], donut_defocus)
show_donut(axes[1, 1], donut_tilt)

fig.savefig(C.FIGDIR / "aberrations_defocus_tilt.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("Astigmatism\n$\\nu = 0,\\, m=2$")
axes[0, 1].set_title("Trefoil\n$\\nu = 0,\\, m=3$")
axes[0, 2].set_title("Quadrafoil\n$\\nu = 0,\\, m=4$")
axes[0, 3].set_title("Pentafoil\n$\\nu = 0,\\, m=5$")

# --- Top row: wavefront-circle schematics --------------------------------
# dp = -grad(Z_j) for the cosine-side primary m-foils: astigmatism Z6, trefoil
# Z10, quadrafoil Z14, pentafoil Z20.  Amplitudes are display scales.
map_circles(0.035, zernike_displacement(6), ax=axes[0, 0])
map_circles(0.0088, zernike_displacement(10), ax=axes[0, 1])
map_circles(0.0040, zernike_displacement(14), ax=axes[0, 2])
map_circles(0.0025, zernike_displacement(20), ax=axes[0, 3])

# --- Bottom row: simulated intra-focal donuts ----------------------------
# The same cosine-side Noll indices, added to the baseline defocus, imprint
# their m-fold shape on the donut.
show_donut(axes[1, 0], sim.simulate_donut(zernikes={6: 7e-6}, defocal_type="intra"))
show_donut(axes[1, 1], sim.simulate_donut(zernikes={10: 2e-6}, defocal_type="intra"))
show_donut(axes[1, 2], sim.simulate_donut(zernikes={14: 1e-6}, defocal_type="intra"))
show_donut(axes[1, 3], sim.simulate_donut(zernikes={20: 1e-6}, defocal_type="intra"))

fig.savefig(C.FIGDIR / "aberrations_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("Spherical\n$\\nu = 1,\\, m=0$")
axes[0, 1].set_title("Coma\n$\\nu = 1,\\, m=1$")

# --- Top row: wavefront-circle schematics --------------------------------
# dp = -grad(Z_j) for spherical (Z11) and cosine coma (Z8).  Spherical moves the
# outer edge in and the inner edge out (thinner ring).  For coma the Z8
# wavefront includes a linear (tilt-like) term; map_circles recenters the view,
# leaving the rho-dependent hole decentering that matters for the donut shape.
map_circles(0.002, zernike_displacement(11), ax=axes[0, 0])
map_circles(0.0126, zernike_displacement(8), ax=axes[0, 1])

# --- Bottom row: simulated intra-focal donuts ----------------------------
# The corresponding positive cosine-side Noll coefficients: spherical Z11 and
# coma Z8.
show_donut(axes[1, 0], sim.simulate_donut(zernikes={11: 3e-7}, defocal_type="intra"))
show_donut(axes[1, 1], sim.simulate_donut(zernikes={8: 2e-6}, defocal_type="intra"))

fig.savefig(C.FIGDIR / "aberrations_spherical_coma.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8), constrained_layout=True, dpi=120)

axes[0, 0].set_title("2nd Astigmatism\n$\\nu = 1,\\, m=2$")
axes[0, 1].set_title("2nd Trefoil\n$\\nu = 1,\\, m=3$")
axes[0, 2].set_title("2nd Quadrafoil\n$\\nu = 1,\\, m=4$")
axes[0, 3].set_title("2nd Pentafoil\n$\\nu = 1,\\, m=5$")

# --- Top row: wavefront-circle schematics --------------------------------
# dp = -grad(Z_j) for the cosine-side secondary m-foils: 2nd astigmatism Z12,
# 2nd trefoil Z18, 2nd quadrafoil Z26, 2nd pentafoil Z34.  Z26 and Z34 are
# display-only higher-Noll modes beyond the paper's Table range.
map_circles(0.0059, zernike_displacement(12), ax=axes[0, 0])
map_circles(0.0023, zernike_displacement(18), ax=axes[0, 1])
map_circles(0.0013, zernike_displacement(26), ax=axes[0, 2])
map_circles(0.00077, zernike_displacement(34), ax=axes[0, 3])

# --- Bottom row: simulated intra-focal donuts ----------------------------
# The same cosine-side secondary m-foil Noll indices; simulate_donut pads the
# Zernike vector for the display-only Z26 and Z34.
show_donut(axes[1, 0], sim.simulate_donut(zernikes={12: 0.75e-6}, defocal_type="intra"))
show_donut(axes[1, 1], sim.simulate_donut(zernikes={18: 0.4e-6}, defocal_type="intra"))
show_donut(axes[1, 2], sim.simulate_donut(zernikes={26: 0.22e-6}, defocal_type="intra"))
show_donut(axes[1, 3], sim.simulate_donut(zernikes={34: 0.12e-6}, defocal_type="intra"))

fig.savefig(C.FIGDIR / "aberrations_2nd_astig_mfoil.pdf", bbox_inches="tight")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(3.5, 3.8), constrained_layout=True, dpi=150)

axes[0, 0].set_title("2nd Spherical\n$\\nu = 2,\\, m=0$")
axes[0, 1].set_title("2nd Coma\n$\\nu = 2,\\, m=1$")

# --- Top row: wavefront-circle schematics --------------------------------
# dp = -grad(Z_j) for secondary spherical (Z22) and cosine secondary coma
# (Z16).  As with primary coma, the Z16 tilt-like term is removed by recentering.
map_circles(0.00066, zernike_displacement(22), ax=axes[0, 0])
map_circles(0.0013, zernike_displacement(16), ax=axes[0, 1])

# --- Bottom row: simulated intra-focal donuts ----------------------------
# The corresponding positive Noll coefficients: secondary spherical Z22 and
# cosine secondary coma Z16.  The spherical coefficient is display-tuned upward
# to better match the old ts_wep.forwardModelPair gallery.
show_donut(axes[1, 0], sim.simulate_donut(zernikes={22: 0.10e-6}, defocal_type="intra"))
show_donut(axes[1, 1], sim.simulate_donut(zernikes={16: 0.2e-6}, defocal_type="intra"))

fig.savefig(C.FIGDIR / "aberrations_2nd_spherical_coma.pdf", bbox_inches="tight")